In [67]:
#README


In [68]:
#IMPORTS

import yfinance as yf
import pandas as pd
import numpy as np
import statsmodels.api as sm




In [69]:
#INPUTS

benchmark = "URTH"
start_date = "2020-12-31"
end_date = "2025-12-31"
interval = "1wk"
return_calc = "linear" #linear / log
beta_adjustment = "blume" #blume / vasicek / none

peer_group = ["ORCL", "PLTR", "PANW", "CRWD", "FTNT"]


In [70]:
#FUNCTIONS

def download_data(tickers: list[str], start_date: str, end_date:str, interval: str):
    """Download data for the given tickers from Yahoo Finance"""
    data = yf.download(tickers = tickers, start = start_date, end = end_date, interval = interval)
    return data

def save_data(data: pd.DataFrame, benchmark: str, peer_group: list[str], start_date: str, end_date: str):
    """Save data to csv file"""
    path = f"./data/{benchmark} + {peer_group}_{start_date}_-_{end_date}.csv"
    data.to_csv(path)

def extract_col(data: pd.DataFrame, field: str):
    """Extract columns from the downloaded dataframe"""
    column_data= data[field]
    return column_data

# def quick_beta(peer, benchmark):
#     cov = np.cov(peer, benchmark)[0, 1]
#     var = np.var(benchmark)
#     return cov / var


In [71]:
#CLOSE_DATA_COLLECTION

ticker_package = peer_group + [benchmark]
data_package = download_data(tickers= ticker_package, start_date= start_date, end_date= end_date, interval= interval)
save_data(data = data_package, benchmark= benchmark, peer_group= peer_group, start_date= start_date, end_date= end_date)
close_data = extract_col(data = data_package, field= "Close")
close_data.head()


[*********************100%***********************]  5 of 6 completed


Ticker,CRWD,FTNT,ORCL,PANW,PLTR,URTH
Date,,,,,,
2020-12-28,52.955002,29.705999,59.762878,59.231667,23.549999,103.073914
2021-01-04,55.932499,29.628000,58.776711,61.091667,25.200001,105.733055
2021-01-11,54.877499,29.306000,57.292904,60.811668,25.639999,104.119209
2021-01-18,55.880001,30.246000,55.976040,60.770000,32.580002,105.769722
2021-01-25,53.950001,28.950001,56.040966,58.458332,35.180000,102.230316


In [72]:
#BASIC_DATA_CLEARING

close_data.columns = close_data.columns.get_level_values(0)
close_data = close_data.dropna()
close_data.head()

Ticker,CRWD,FTNT,ORCL,PANW,PLTR,URTH
Date,,,,,,
2020-12-28,52.955002,29.705999,59.762878,59.231667,23.549999,103.073914
2021-01-04,55.932499,29.628000,58.776711,61.091667,25.200001,105.733055
2021-01-11,54.877499,29.306000,57.292904,60.811668,25.639999,104.119209
2021-01-18,55.880001,30.246000,55.976040,60.770000,32.580002,105.769722
2021-01-25,53.950001,28.950001,56.040966,58.458332,35.180000,102.230316


In [73]:
#LOG_RETURN_CALCULATION
if return_calc == "log":
    return_data = np.log(close_data / close_data.shift(1))
elif return_calc == "linear":
    return_data = (close_data / close_data.shift(1)) -1
else:
    raise ValueError("Return calculation must be either 'log' or 'linear'")

return_data = return_data.dropna()
return_data.head()

Ticker,CRWD,FTNT,ORCL,PANW,PLTR,URTH
Date,,,,,,
2021-01-04,0.056227,-0.002626,-0.016501,0.031402,0.070064,0.025798
2021-01-11,-0.018862,-0.010868,-0.025245,-0.004583,0.017460,-0.015263
2021-01-18,0.018268,0.032075,-0.022985,-0.000685,0.270671,0.015852
2021-01-25,-0.034538,-0.042849,0.001160,-0.038040,0.079804,-0.033463
2021-02-01,0.035820,0.074128,0.052457,0.082965,-0.032121,0.042246


In [74]:
#STATSMODELS_REGRESSION

market = return_data[benchmark]
raw_betas = []
std_errors = []

for ticker in peer_group:
    y = return_data[ticker]
    x = sm.add_constant(market)
    model = sm.OLS(y, x).fit()

    raw_betas.append(model.params[benchmark])
    std_errors.append(model.bse[benchmark])

beta_results = pd.DataFrame({
    "Ticker" : peer_group,
    "Standard Error" : std_errors,
    "Raw Betas" : raw_betas,
})

beta_results
# print(model.params["URTH"]) #raw_beta
# print(model.bse["URTH"]) #standard_error



,Ticker,Standard Error,Raw Betas
0,ORCL,0.123742,1.187064
1,PLTR,0.233816,2.105782
2,PANW,0.134381,1.219595
3,CRWD,0.162595,1.684180
4,FTNT,0.137358,1.435668


In [75]:
#BETA_ADJUSTMENT

beta_mean = beta_results["Raw Betas"].mean()
beta_cross_var = beta_results["Raw Betas"].var(ddof=1)

vas_beta = []

for ticker in beta_results["Ticker"]:
    se = beta_results.loc[beta_results["Ticker"] == ticker, "Standard Error"].item()
    weight = beta_cross_var / (beta_cross_var + (se ** 2))
    r_bet = beta_results.loc[beta_results["Ticker"] == ticker, "Raw Betas"].item()
    adj_beta = weight * r_bet + (1 - weight) * beta_mean
    vas_beta.append(adj_beta)

beta_results["Blume adj. beta"] = beta_results["Raw Betas"] * (2/3) + (1 * (1/3))
beta_results["Vasicek adj. beta"] = vas_beta

beta_results


,Ticker,Standard Error,Raw Betas,Blume adj. beta,Vasicek adj. beta
0,ORCL,0.123742,1.187064,1.124709,1.219578
1,PLTR,0.233816,2.105782,1.737188,1.946782
2,PANW,0.134381,1.219595,1.146397,1.253679
3,CRWD,0.162595,1.684180,1.456120,1.659790
4,FTNT,0.137358,1.435668,1.290445,1.446152
